# Popularity Baseline (Implicit Ranking)
Run these cells top-to-bottom to reproduce the popularity baseline.

In [ ]:
# Install dependencies (Colab)
!pip install -q numpy scipy scikit-learn

import csv
import json
import math
import os
from datetime import datetime
import numpy as np
from scipy import sparse

def load_sparse_matrix(path: str) -> sparse.csr_matrix:
    matrix = sparse.load_npz(path)
    if not sparse.isspmatrix_csr(matrix):
        matrix = matrix.tocsr()
    return matrix

def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _top_k_indices(scores: np.ndarray, k: int) -> np.ndarray:
    if k <= 0:
        return np.array([], dtype=np.int64)
    if k >= scores.shape[0]:
        return np.argsort(-scores)
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(-scores[idx])]

def compute_user_metrics(
    scores: np.ndarray,
    train_items: np.ndarray,
    test_items: np.ndarray,
    k: int,
    use_ndcg: bool = True,
):
    if test_items.size == 0 or k <= 0 or scores.size == 0:
        return None
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    ranked = _top_k_indices(scores, k)
    test_set = set(test_items.tolist())
    hits = sum(1 for item in ranked if item in test_set)
    precision = hits / k
    recall = hits / len(test_set)
    ndcg = 0.0
    if use_ndcg:
        dcg = 0.0
        for rank, item in enumerate(ranked):
            if item in test_set:
                dcg += 1.0 / math.log2(rank + 2)
        ideal = min(len(test_set), k)
        idcg = sum(1.0 / math.log2(rank + 2) for rank in range(ideal))
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
    return precision, recall, ndcg

def evaluate_ranking(
    score_fn,
    train_interactions: sparse.csr_matrix,
    test_interactions: sparse.csr_matrix,
    k: int,
    use_ndcg: bool = True,
) -> dict:
    precisions = []
    recalls = []
    ndcgs = []
    num_users = train_interactions.shape[0]
    for user_id in range(num_users):
        test_items = test_interactions[user_id].indices
        if test_items.size == 0:
            continue
        train_items = train_interactions[user_id].indices
        scores = score_fn(user_id)
        metrics = compute_user_metrics(scores, train_items, test_items, k, use_ndcg)
        if metrics is None:
            continue
        precision, recall, ndcg = metrics
        precisions.append(precision)
        recalls.append(recall)
        if use_ndcg:
            ndcgs.append(ndcg)
    result = {
        "k": float(k),
        "users_evaluated": float(len(precisions)),
        "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
    }
    if use_ndcg:
        result["ndcg_at_k"] = float(np.mean(ndcgs)) if ndcgs else 0.0
    return result

def recommend_top_k(scores: np.ndarray, train_items: np.ndarray, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    scores = scores.copy()
    scores[train_items] = -np.inf
    k = min(k, scores.size)
    return _top_k_indices(scores, k)

In [ ]:
# Config (match popularity_baseline.py defaults)
DATA_DIR_P5 = "phase5_outputs"
DATA_DIR_P4 = "phase4_outputs"
OUTPUT_DIR = os.path.join("outputs", f"popularity_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
K = 10
TOP_N = 20
EXAMPLE_USERS = 5
RANDOM_STATE = 42
USE_NDCG = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "train_interactions.npz"))
test_interactions = load_sparse_matrix(os.path.join(DATA_DIR_P5, "test_interactions.npz"))

print("Train shape:", train_interactions.shape)
print("Test shape:", test_interactions.shape)

In [ ]:
# Evaluate + save outputs
popularity = np.asarray(train_interactions.getnnz(axis=0)).ravel().astype(float)

def score_fn(_: int) -> np.ndarray:
    return popularity

metrics = evaluate_ranking(
    score_fn,
    train_interactions,
    test_interactions,
    k=K,
    use_ndcg=USE_NDCG,
 )

print("Popularity baseline metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "metrics.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics.keys()))
    writer.writeheader()
    writer.writerow(metrics)

top_n = min(TOP_N, popularity.size)
top_items = np.argsort(-popularity)[:top_n]

book2id_path = os.path.join(DATA_DIR_P4, "book2id.json")
id2book = {}
if os.path.exists(book2id_path):
    id2book = {v: k for k, v in load_json(book2id_path).items()}

with open(os.path.join(OUTPUT_DIR, "top_popular_books.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["item_id", "book_id", "train_interactions"])
    for item_id in top_items:
        book_id = id2book.get(int(item_id), "")
        writer.writerow([int(item_id), book_id, int(popularity[item_id])])

user2id_path = os.path.join(DATA_DIR_P4, "user2id.json")
id2user = {}
if os.path.exists(user2id_path):
    id2user = {v: k for k, v in load_json(user2id_path).items()}

rng = np.random.RandomState(RANDOM_STATE)
users_with_test = np.where(test_interactions.getnnz(axis=1) > 0)[0]
if users_with_test.size > 0:
    sampled_users = rng.choice(
        users_with_test,
        size=min(EXAMPLE_USERS, users_with_test.size),
        replace=False,
    )
else:
    sampled_users = np.array([], dtype=int)

with open(os.path.join(OUTPUT_DIR, "recommendation_examples.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["user_id", "user_index", "recommended_item_ids", "recommended_book_ids"])
    for user_id in sampled_users:
        train_items = train_interactions[user_id].indices
        topk = recommend_top_k(popularity, train_items, K)
        rec_items = [str(int(i)) for i in topk]
        rec_books = [id2book.get(int(i), "") for i in topk]
        writer.writerow([
            id2user.get(int(user_id), ""),
            int(user_id),
            " ".join(rec_items),
            " ".join(rec_books),
        ])

print(f"Outputs saved to: {OUTPUT_DIR}")

In [ ]:
# Save a compact CSV summary for quick comparison
summary_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
summary_row = {
    "run_id": os.path.basename(OUTPUT_DIR),
    "k": metrics.get("k"),
    "users_evaluated": metrics.get("users_evaluated"),
    "precision_at_k": metrics.get("precision_at_k"),
    "recall_at_k": metrics.get("recall_at_k"),
    "ndcg_at_k": metrics.get("ndcg_at_k", None),
}

write_header = not os.path.exists(summary_path)
with open(summary_path, "a", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_row.keys()))
    if write_header:
        writer.writeheader()
    writer.writerow(summary_row)

print(f"Summary saved to: {summary_path}")